# AI-based Hyperlocal Demand Prediction — Dark Stores & Quick Commerce
## Stage 1: Latent Demand Recovery (XGBoost) — v1

Direct port of the LightGBM v5 pipeline to XGBoost for comparison purposes.

| Design choice | LightGBM v5 | XGBoost v1 (this notebook) |
|---|---|---|
| Objective | `tweedie` (power=1.5) | `reg:tweedie` (power=1.5) |
| Early stopping metric | Custom WAPE feval | Custom WAPE `custom_metric` |
| Feature importance | `feature_importance(type='gain')` | `get_score(importance_type='gain')` |
| Categorical encoding | label-encoded int32 | label-encoded int32 (same) |
| Power-law weights | `plw()` passed as `weight=` | `plw()` passed as `weight=` |
| Calibration | scalar factor on in-stock val rows | identical |
| Everything else | unchanged | unchanged |

**Key XGBoost translation notes:**
- `lgb.train(params, dtrain, feval=...)` → `xgb.train(params, dtrain, custom_metric=...)`
- `model.feature_importance(type='gain')` → `model.get_score(importance_type='gain')` (returns dict; must align to FEAT_COLS)
- XGBoost `early_stopping_rounds` lives in `xgb.train(...)` not in params
- `model.best_iteration` → `model.best_iteration` (same attribute name)
- `model.predict(X, num_iteration=N)` → `model.predict(dmat, iteration_range=(0, N))`
- LGB `num_boost_round` param key → XGB `num_boost_round` argument (same name)


## 1 · Setup

In [4]:
import ast, gc, logging, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

OUTPUT_DIR = Path("/kaggle/working/stage1_xgb_v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def mem_mb():
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024
    except Exception:
        pass
    return 0.0

print(f"XGBoost version : {xgb.__version__}")
print(f"Setup complete   RAM: {mem_mb():.0f} MB")

XGBoost version : 3.2.0
Setup complete   RAM: 254 MB


## 2 · Config

In [5]:
class C:
    CITY_ID      = "city_id";     STORE_ID    = "store_id"
    MGMT_GRP     = "management_group_id"
    CAT1         = "first_category_id";  CAT2 = "second_category_id"
    CAT3         = "third_category_id";  PRODUCT_ID = "product_id"
    DT           = "dt"
    SALE_AMOUNT  = "sale_amount"        # daily total observed sales
    STOCK_HR_CNT = "stock_hour6_22_cnt" # STOCKOUT hours 6AM-10PM (0=fully stocked)
    HOURS_SALE   = "hours_sale"         # 24-element hourly sales array
    HOURS_STOCK  = "hours_stock_status" # 24-element: 1=stockout 0=in-stock
    DISCOUNT     = "discount";  ACTIVITY = "activity_flag"
    HOLIDAY      = "holiday_flag"
    PRECPT       = "precpt";    TEMP     = "avg_temperature"
    HUMID        = "avg_humidity";  WIND = "avg_wind_level"
    # Derived
    IN_STOCK     = "in_stock"           # 1 = zero stockout hours
    SALE_LAG     = "sale_for_lag"       # NaN on censored days
    SR_I         = "stockout_ratio"
    MU_I         = "mean_sales_mu"
    D_HAT        = "pred_demand"
    D_T          = "recovered_demand"

class P:
    POWER_LAW_ALPHA  = 2.83   # Section 3.2
    TOP_SKU_FRAC     = 0.20
    HOLIDAY_UPLIFT   = 1.27
    PROMO_MEDIAN     = 1.43;  PROMO_IQR_LOW = 1.05;  PROMO_IQR_HIGH = 2.06
    RAIN_VEG_EFFECT  = 0.11
    BETA_DISC_SO     = -0.046   # beta: discount -> stockout p=0.018
    BETA_RAIN_SO     =  0.108   # beta: rainfall -> stockout p=0.05
    BETA_TEMP_FROZEN =  0.065   # beta: temp -> frozen SO p=0.024
    BETA_TEMP_MEAT   = -0.067   # beta: temp -> meat SO p=0.033
    TARGET_RHO_DS    =  0.10
    TARGET_WPE       =  0.03

OPERATING_HOURS  = 16   # 6AM-10PM
IN_STOCK_MAX_SO  = 0
IN_STOCK_THRESH  = 2    # allow up to 2 stockout hours

MNAR_RATE   = 0.15
LAG_DAYS    = [1, 7, 14, 28]
ROLL_WINS   = [7, 14, 30]
VAL_RATIO   = 0.20
RANDOM_SEED = 42

# ── XGBoost params (Tweedie, equivalent to LightGBM config) ──────────────
# reg:tweedie with tweedie_variance_power=1.5 mirrors LGB objective=tweedie power=1.5
# max_leaves=127 mirrors num_leaves=127 (XGBoost uses max_leaves for leaf-wise growth)
# colsample_bytree=0.8 mirrors feature_fraction=0.8
# subsample=0.8 + subsample_freq=5 mirrors bagging_fraction=0.8, bagging_freq=5
# reg_alpha/reg_lambda mirror lambda_l1/lambda_l2
# max_bin=127 mirrors max_bin=127
XGB_PARAMS = {
    "objective":                "reg:tweedie",
    "tweedie_variance_power":   1.5,          # from paper power-law alpha=2.83
    "tree_method":              "hist",        # fastest; equivalent to LGB's histogram method
    "grow_policy":              "lossguide",  # leaf-wise growth, matches LGB default
    "max_leaves":               127,           # mirrors num_leaves=127
    "learning_rate":            0.05,
    "min_child_weight":         1e-3,          # mirrors min_child_weight
    "colsample_bytree":         0.8,           # mirrors feature_fraction
    "subsample":                0.8,           # mirrors bagging_fraction
    "subsample_freq":           5,             # mirrors bagging_freq (alias: n_skip_drop)
    # NOTE: subsample_freq is only respected when subsample < 1.0 in hist mode
    "reg_alpha":                0.1,           # L1, mirrors lambda_l1
    "reg_lambda":               0.1,           # L2, mirrors lambda_l2
    "max_bin":                  127,           # mirrors max_bin
    "seed":                     RANDOM_SEED,
    "nthread":                  -1,
    "verbosity":                0,             # silent
}

NUM_BOOST_ROUND = 4000
EARLY_STOPPING  = 150   # rounds without WAPE improvement before stopping
VERBOSE_EVAL    = 100   # print every N rounds

print("Config loaded")
print(f"  Tweedie power   : {XGB_PARAMS['tweedie_variance_power']} (from alpha={P.POWER_LAW_ALPHA})")
print(f"  IN_STOCK defined: stock_hour6_22_cnt <= {IN_STOCK_THRESH} (stockout hours)")

Config loaded
  Tweedie power   : 1.5 (from alpha=2.83)
  IN_STOCK defined: stock_hour6_22_cnt <= 2 (stockout hours)


## 3 · Dataset Paths

In [6]:
import glob, time

found = sorted(glob.glob("/kaggle/input/**/*.parquet", recursive=True))
print("Parquet files:")
for f in found: print(f"  {f}")

TRAIN_PATH = next((f for f in found if "train" in Path(f).name.lower()), None)
EVAL_PATH  = next((f for f in found if "eval"  in Path(f).name.lower()), None)

# Override manually if needed:
# TRAIN_PATH = "/kaggle/input/.../train.parquet"
# EVAL_PATH  = "/kaggle/input/.../eval.parquet"

Parquet files:
  /kaggle/input/datasets/dangtrannhuminh/datastorm/eval.parquet
  /kaggle/input/datasets/dangtrannhuminh/datastorm/train.parquet


## 4 · Load & Parse

`hours_stock_status` encoding (confirmed from dataset):
- `1` = stockout hour, `0` = in-stock hour
- `stock_hour6_22_cnt` = count of stockout hours (not in-stock hours)
- `in_stock = (stock_hour6_22_cnt <= 2)` — at most 2 stockout hours = clean day

In [7]:
LOAD_COLS = [
    C.CITY_ID, C.STORE_ID, C.MGMT_GRP,
    C.CAT1, C.CAT2, C.CAT3, C.PRODUCT_ID, C.DT,
    C.SALE_AMOUNT, C.STOCK_HR_CNT,
    C.HOURS_SALE, C.HOURS_STOCK,
    C.DISCOUNT, C.ACTIVITY, C.HOLIDAY,
    C.PRECPT, C.TEMP, C.HUMID, C.WIND,
]

def parse_arr(val, n=24, fill=0.0):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.full(n, fill, np.float32)
    if isinstance(val, (list, np.ndarray)):
        a = np.array(val, np.float32)
        if len(a) >= n: return a[:n]
        return np.pad(a, (0, n-len(a)), constant_values=fill)
    if isinstance(val, str):
        try:
            import ast as _ast
            a = np.array(_ast.literal_eval(val.strip()), np.float32)
            if len(a) >= n: return a[:n]
            return np.pad(a, (0, n-len(a)), constant_values=fill)
        except Exception:
            pass
    return np.full(n, fill, np.float32)

def extract_hourly_scalars(df):
    """
    Parse hourly arrays ONCE into numpy matrices, then use vectorised
    column ops.

    hours_stock_status encoding (confirmed from dataset):
      1 = stockout hour,  0 = in-stock hour

    Causality rule:
      Supply-side (hours_stock_status) → safe for current day
      Demand-side (hours_sale)         → shift by 1 day before use
    """
    N = len(df)
    print(f"  Vectorised array parsing on {N:,} rows ...")

    stock_mat = np.vstack(df[C.HOURS_STOCK].apply(
        lambda v: parse_arr(v, 24, 0.0)).values).astype(np.int8)   # 1=stockout
    sale_mat  = np.vstack(df[C.HOURS_SALE].apply(
        lambda v: parse_arr(v, 24, 0.0)).values).astype(np.float32)

    # Supply-side scalars (safe for current day)
    op_stock  = stock_mat[:, 6:22]
    df["hours_in_stock_6_22"] = (op_stock == 0).sum(axis=1).astype(np.int8)

    first_so = np.full(N, -1, dtype=np.int8)
    for h in range(6, 22):
        mask = (first_so == -1) & (stock_mat[:, h] == 1)
        first_so[mask] = h
    df["first_stockout_hour"] = first_so

    df["stockout_at_9am"]  = (stock_mat[:, 9]  == 1).astype(np.int8)
    df["stockout_at_4pm"]  = (stock_mat[:, 16] == 1).astype(np.int8)
    df["stockout_in_peak"] = ((stock_mat[:, 9] == 1) |
                               (stock_mat[:, 16] == 1)).astype(np.int8)

    # Demand-side scalars (will be lagged by 1 day)
    op_sale   = sale_mat[:, 6:22]
    peak_idx  = np.argmax(op_sale, axis=1)
    df["_peak_sale_hour_raw"] = (peak_idx + 6).astype(np.int8)
    morning   = sale_mat[:, 6:12].sum(axis=1)
    total     = op_sale.sum(axis=1) + 1e-8
    df["_morning_share_raw"]  = (morning / total).astype(np.float32)

    del stock_mat, sale_mat, op_stock, op_sale
    return df

def load_daily(path, mode="train"):
    t0    = time.time()
    avail = pd.read_parquet(path, columns=None).columns.tolist()
    cols  = [c for c in LOAD_COLS if c in avail]
    df    = pd.read_parquet(path, columns=cols)

    df[C.DT] = pd.to_datetime(df[C.DT])
    for col in [C.SALE_AMOUNT, C.DISCOUNT, C.PRECPT, C.TEMP, C.HUMID, C.WIND, C.STOCK_HR_CNT]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in [C.HOLIDAY, C.ACTIVITY]:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)

    df[C.SALE_AMOUNT]  = df[C.SALE_AMOUNT].fillna(0).clip(lower=0)
    df[C.STOCK_HR_CNT] = df[C.STOCK_HR_CNT].fillna(OPERATING_HOURS).clip(0, OPERATING_HOURS).astype(np.int8)
    df[C.IN_STOCK]     = (df[C.STOCK_HR_CNT] <= IN_STOCK_THRESH).astype(np.int8)

    df = df.sort_values([C.STORE_ID, C.PRODUCT_ID, C.DT]).reset_index(drop=True)

    if C.HOURS_SALE in df.columns and C.HOURS_STOCK in df.columns:
        df = extract_hourly_scalars(df)
        df.drop(columns=[C.HOURS_SALE, C.HOURS_STOCK], inplace=True)
    else:
        for col in ["hours_in_stock_6_22", "first_stockout_hour",
                    "stockout_at_9am", "stockout_at_4pm", "stockout_in_peak",
                    "_peak_sale_hour_raw", "_morning_share_raw"]:
            df[col] = 0

    log.info(f"[{mode}] {len(df):,} rows in {time.time()-t0:.1f}s  RAM:{mem_mb():.0f}MB")
    log.info(f"  Date: {df[C.DT].min().date()} to {df[C.DT].max().date()}")
    log.info(f"  Stores: {df[C.STORE_ID].nunique()}  Products: {df[C.PRODUCT_ID].nunique()}")
    log.info(f"  In-stock days (stockout_hrs <= {IN_STOCK_THRESH}): {df[C.IN_STOCK].mean():.1%}")
    return df

print("Loading train ...")
df = load_daily(TRAIN_PATH, "train")
print(f"Shape: {df.shape}   RAM: {mem_mb():.0f} MB")
print(f"\nIn-stock sanity:")
print(f"  in_stock=1 rows : {df[C.IN_STOCK].sum():,}  ({df[C.IN_STOCK].mean():.1%})")
print(f"  Mean sale (in-stock) : {df[df[C.IN_STOCK]==1][C.SALE_AMOUNT].mean():.4f}")
print(f"  Mean sale (censored) : {df[df[C.IN_STOCK]==0][C.SALE_AMOUNT].mean():.4f}")
print("  In-stock mean MUST be higher than censored mean")

Loading train ...
  Vectorised array parsing on 4,500,000 rows ...


10:11:13  INFO      [train] 4,500,000 rows in 57.9s  RAM:4458MB
10:11:13  INFO        Date: 2024-03-28 to 2024-06-25
10:11:13  INFO        Stores: 898  Products: 865
10:11:13  INFO        In-stock days (stockout_hrs <= 2): 61.8%


Shape: (4500000, 25)   RAM: 4458 MB

In-stock sanity:
  in_stock=1 rows : 2,780,661  (61.8%)
  Mean sale (in-stock) : 1.0171
  Mean sale (censored) : 0.9687
  In-stock mean MUST be higher than censored mean


## 5 · Feature Engineering

Identical to LightGBM v5 — no changes. Lag and rolling statistics use `sale_for_lag`
(NaN on censored days). NaN lags are filled with the 14-day rolling mean.

Hourly-derived features: only supply-side features from `hours_stock_status` are used
for the current day. Demand-side features from `hours_sale` are shifted by 1 day.

In [8]:
def build_features(df, is_train=True):
    t0  = time.time()
    grp = [C.STORE_ID, C.PRODUCT_ID]

    # sale_for_lag: NaN on censored days
    df[C.SALE_LAG] = df[C.SALE_AMOUNT].where(df[C.IN_STOCK] == 1, other=np.nan)

    # Calendar (c in paper)
    df["feat_dow"]       = df[C.DT].dt.dayofweek.astype(np.int8)
    df["feat_is_wkend"]  = (df["feat_dow"] >= 5).astype(np.int8)
    df["feat_month"]     = df[C.DT].dt.month.astype(np.int8)
    df["feat_woy"]       = df[C.DT].dt.isocalendar().week.astype(np.int16)
    df["feat_dom"]       = df[C.DT].dt.day.astype(np.int8)
    df["feat_holiday"]   = df[C.HOLIDAY].fillna(0).astype(np.int8) if C.HOLIDAY in df.columns else np.int8(0)
    df["feat_dow_sin"]   = np.sin(2*np.pi*df["feat_dow"]/7).astype(np.float32)
    df["feat_dow_cos"]   = np.cos(2*np.pi*df["feat_dow"]/7).astype(np.float32)
    df["feat_month_sin"] = np.sin(2*np.pi*df["feat_month"]/12).astype(np.float32)
    df["feat_month_cos"] = np.cos(2*np.pi*df["feat_month"]/12).astype(np.float32)

    # Promotional (p in paper)
    disc = df[C.DISCOUNT].fillna(0).clip(0,1).astype(np.float32) if C.DISCOUNT in df.columns \
           else pd.Series(0.0, index=df.index, dtype=np.float32)
    act  = df[C.ACTIVITY].fillna(0).astype(np.int8) if C.ACTIVITY in df.columns \
           else pd.Series(0, index=df.index, dtype=np.int8)
    df["feat_discount"]     = disc
    df["feat_activity"]     = act
    df["feat_promo_uplift"] = np.where(act==1,
        np.clip(P.PROMO_MEDIAN+(disc-0.20)*(P.PROMO_IQR_HIGH-P.PROMO_MEDIAN),
                P.PROMO_IQR_LOW, P.PROMO_IQR_HIGH), 1.0).astype(np.float32)
    df["feat_disc_so_risk"] = (P.BETA_DISC_SO * disc).astype(np.float32)
    df["feat_disc_sq"]      = (disc**2).astype(np.float32)

    # Weather (w in paper)
    precpt = df[C.PRECPT].fillna(0).clip(0).astype(np.float32) if C.PRECPT in df.columns \
             else pd.Series(0.0, index=df.index, dtype=np.float32)
    temp   = df[C.TEMP].fillna(20).astype(np.float32) if C.TEMP in df.columns \
             else pd.Series(20.0, index=df.index, dtype=np.float32)
    df["feat_precpt"]    = precpt
    df["feat_temp"]      = temp
    df["feat_humid"]     = df[C.HUMID].fillna(60).astype(np.float32) if C.HUMID in df.columns else np.float32(60)
    df["feat_wind"]      = df[C.WIND].fillna(2).astype(np.float32)   if C.WIND  in df.columns else np.float32(2)
    df["feat_rain_dem"]  = (P.RAIN_VEG_EFFECT * np.log1p(precpt)).astype(np.float32)
    df["feat_rain_so"]   = (P.BETA_RAIN_SO    * np.log1p(precpt)).astype(np.float32)
    df["feat_is_rainy"]  = (precpt > 5.0).astype(np.int8)
    df["feat_t_frozen"]  = (P.BETA_TEMP_FROZEN * temp).astype(np.float32)
    df["feat_t_meat"]    = (P.BETA_TEMP_MEAT   * temp).astype(np.float32)

    # Supply context (BLINDFOLDED — _tmp_ prefix, excluded from FEAT_COLS)
    sc = df[C.STOCK_HR_CNT].astype(np.float32)
    df["_tmp_so_hours"]      = sc
    df["_tmp_so_frac"]       = (sc / OPERATING_HOURS).astype(np.float32)
    df["_tmp_stock_frac"]    = (1.0 - sc/OPERATING_HOURS).astype(np.float32)
    df["_tmp_in_stock"]      = df[C.IN_STOCK].astype(np.int8)

    # Lagged supply (SAFE — yesterday's stockout info)
    df["feat_in_stock_lag1"] = df.groupby(grp)[C.IN_STOCK].shift(1).fillna(1).astype(np.int8)
    df["feat_so_frac_lag1"]  = df.groupby(grp)["_tmp_so_frac"].shift(1).fillna(0).astype(np.float32)

    # Hourly supply-side (BLINDFOLDED)
    df["_tmp_hours_in_stock"]   = df["hours_in_stock_6_22"].astype(np.int8)
    df["_tmp_first_so_hour"]    = df["first_stockout_hour"].astype(np.int8)
    df["_tmp_so_at_9am"]        = df["stockout_at_9am"].astype(np.int8)
    df["_tmp_so_at_4pm"]        = df["stockout_at_4pm"].astype(np.int8)
    df["_tmp_so_in_peak"]       = df["stockout_in_peak"].astype(np.int8)

    # Hourly supply-side (SAFE LAGGED)
    df["feat_so_at_9am_lag1"]   = df.groupby(grp)["stockout_at_9am"].shift(1).fillna(0).astype(np.int8)
    df["feat_so_at_4pm_lag1"]   = df.groupby(grp)["stockout_at_4pm"].shift(1).fillna(0).astype(np.int8)

    # Hourly demand-side (LAGGED — causality safe)
    df["feat_peak_hr_lag1"]     = (df.groupby(grp)["_peak_sale_hour_raw"]
                                      .shift(1).fillna(9).astype(np.int8))
    df["feat_morning_sh_lag1"]  = (df.groupby(grp)["_morning_share_raw"]
                                      .shift(1).fillna(0.4).astype(np.float32))

    # Lag features (on sale_for_lag — NaN on censored days)
    roll14 = (df.groupby(grp)[C.SALE_LAG]
                .transform(lambda x: x.shift(1).rolling(14, min_periods=1).mean()))
    for lag in LAG_DAYS:
        raw_lag = df.groupby(grp)[C.SALE_LAG].shift(lag)
        df[f"feat_lag_{lag}d"] = raw_lag.fillna(roll14).fillna(0).astype(np.float32)

    # Rolling features (on sale_for_lag)
    for w in ROLL_WINS:
        df[f"feat_rmean_{w}d"] = (df.groupby(grp)[C.SALE_LAG]
                                     .transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
                                     .fillna(0).astype(np.float32))
        df[f"feat_rstd_{w}d"]  = (df.groupby(grp)[C.SALE_LAG]
                                     .transform(lambda x: x.shift(1).rolling(w, min_periods=1).std().fillna(0))
                                     .fillna(0).astype(np.float32))
        df[f"feat_rmax_{w}d"]  = (df.groupby(grp)[C.SALE_LAG]
                                     .transform(lambda x: x.shift(1).rolling(w, min_periods=1).max())
                                     .fillna(0).astype(np.float32))
    df["feat_wday_hist"] = (df.groupby([C.STORE_ID, C.PRODUCT_ID, "feat_dow"])[C.SALE_LAG]
                               .transform(lambda x: x.shift(1).expanding().mean())
                               .fillna(0).astype(np.float32))

    # Global baseline features
    in_stk    = df[df[C.IN_STOCK] == 1]
    sku_mu    = in_stk.groupby(C.PRODUCT_ID)[C.SALE_AMOUNT].mean().rename("sku_global_mean")
    store_mu  = in_stk.groupby(C.STORE_ID)[C.SALE_AMOUNT].mean().rename("store_global_mean")
    overall_sku_mean   = float(in_stk[C.SALE_AMOUNT].mean()) if len(in_stk) else 0.0
    overall_store_mean = overall_sku_mean
    df = df.merge(sku_mu.reset_index(),   on=C.PRODUCT_ID, how="left")
    df = df.merge(store_mu.reset_index(), on=C.STORE_ID,   how="left")
    df["feat_sku_global_mean"]   = df["sku_global_mean"].fillna(overall_sku_mean).astype(np.float32)
    df["feat_store_global_mean"] = df["store_global_mean"].fillna(overall_store_mean).astype(np.float32)
    df.drop(columns=["sku_global_mean","store_global_mean"], inplace=True)

    # Entity / pair stats
    mu = df.groupby(grp)[C.SALE_LAG].transform("mean").fillna(0).astype(np.float32)
    df[C.MU_I]      = mu
    df["feat_mu_i"] = mu

    avail = df.groupby(grp)[C.IN_STOCK].transform("mean").astype(np.float32)
    df["feat_availability_ratio"] = avail

    eps = 1e-4
    df["feat_demand_uplift_prior"] = (mu / (avail + eps)).astype(np.float32)

    sku_mu2  = df.groupby(grp)[C.SALE_LAG].mean()
    thresh   = sku_mu2.quantile(1.0 - P.TOP_SKU_FRAC)
    top_df   = sku_mu2[sku_mu2 >= thresh].reset_index()[[C.STORE_ID, C.PRODUCT_ID]]
    top_df["feat_is_top_sku"] = np.int8(1)
    df = df.merge(top_df, on=grp, how="left")
    df["feat_is_top_sku"] = df["feat_is_top_sku"].fillna(0).astype(np.int8)

    # Label encode categoricals
    for col in [C.CITY_ID, C.STORE_ID, C.MGMT_GRP, C.CAT1, C.CAT2, C.CAT3, C.PRODUCT_ID]:
        if col in df.columns:
            df[f"feat_{col}"] = df[col].astype("category").cat.codes.astype(np.int32)

    # Interactions
    df["feat_disc_x_temp"]    = (disc * temp).astype(np.float32)
    df["feat_promo_x_rain"]   = (act  * precpt).astype(np.float32)
    df["feat_wkend_x_promo"]  = (df["feat_is_wkend"] * act).astype(np.int8)
    df["feat_holiday_x_act"]  = (df["feat_holiday"]  * act).astype(np.int8)
    df["_tmp_so_x_peak"]      = (df["_tmp_so_in_peak"] * df["_tmp_so_frac"]).astype(np.float32)
    eps2 = 1e-6
    df["feat_cv_14d"]         = (df["feat_rstd_14d"] / (df["feat_rmean_14d"] + eps2)).astype(np.float32)

    n_feat = len([c for c in df.columns if c.startswith("feat_")])
    log.info(f"Features built: {n_feat} in {time.time()-t0:.1f}s  RAM:{mem_mb():.0f}MB")
    return df

df = build_features(df, is_train=True)
FEAT_COLS = sorted([c for c in df.columns if c.startswith("feat_")])

print(f"Total features : {len(FEAT_COLS)}")
print(f"RAM            : {mem_mb():.0f} MB")
print(f"NaN in features: {df[FEAT_COLS].isna().sum().sum()} (should be 0)")

10:15:36  INFO      Features built: 62 in 261.6s  RAM:7378MB


Total features : 62
RAM            : 6990 MB
NaN in features: 0 (should be 0)


## 6 · MNAR Simulation

In [9]:
rng = np.random.default_rng(RANDOM_SEED)

candidate  = (df[C.IN_STOCK] == 1) & (df[C.SALE_AMOUNT] > 0)
rand_draw  = rng.random(len(df))
syn_mask   = candidate.values & (rand_draw < MNAR_RATE)

df["synth_mask"]   = syn_mask.astype(np.int8)
df["train_target"] = np.where(syn_mask, df[C.SALE_AMOUNT].values, np.nan)

n_cand = candidate.sum()
n_mask = syn_mask.sum()
print(f"Candidate days (in-stock, demand>0): {n_cand:,}  ({n_cand/len(df):.1%})")
print(f"Synthetically masked (targets)     : {n_mask:,}  ({n_mask/n_cand:.1%} of candidates)")
print(f"All targets > 0: {(df.loc[syn_mask, C.SALE_AMOUNT] > 0).all()}")

tgt = df.loc[syn_mask, C.SALE_AMOUNT]
print(f"Target stats — min:{tgt.min():.3f}  mean:{tgt.mean():.3f}  max:{tgt.max():.3f}")

Candidate days (in-stock, demand>0): 2,732,168  (60.7%)
Synthetically masked (targets)     : 409,855  (15.0% of candidates)
All targets > 0: True
Target stats — min:0.060  mean:1.038  max:44.900


## 7 · Temporal Split

In [10]:
target_df  = df[df["synth_mask"] == 1].sort_values(C.DT)
cut_idx    = int(len(target_df) * (1.0 - VAL_RATIO))
split_date = target_df[C.DT].iloc[cut_idx].normalize()

train_df   = target_df[target_df[C.DT] <= split_date]
val_df     = target_df[target_df[C.DT] >  split_date]

print(f"Split date  : {split_date.date()}")
print(f"Train range : {train_df[C.DT].min().date()} to {train_df[C.DT].max().date()}")
print(f"Val range   : {val_df[C.DT].min().date()} to {val_df[C.DT].max().date()}")
print(f"Train rows  : {len(train_df):,}    Val rows: {len(val_df):,}")

Split date  : 2024-06-08
Train range : 2024-03-28 to 2024-06-08
Val range   : 2024-06-09 to 2024-06-25
Train rows  : 330,406    Val rows: 79,449


## 8 · Train XGBoost (Tweedie)

**Translation notes vs LightGBM v5:**

- **Metric**: LGB used `metric='None'` + custom `feval` to prevent built-in loss from driving early stopping. XGBoost's equivalent is omitting `eval_metric` from params and providing `custom_metric` to `xgb.train()`. The `custom_metric` function drives both logging and early stopping.
- **Custom metric signature**: XGBoost `custom_metric(y_pred, dmatrix)` — note the argument order is **reversed** from LightGBM's `feval(y_pred, dataset)`. Also, XGBoost expects the function to return `(name, value)` where **lower is better by default**; pass `maximize=False` (the default) since WAPE should be minimised.
- **Early stopping**: Passed as `early_stopping_rounds=N` argument to `xgb.train()`, not inside params.
- **Prediction**: `model.predict(dmatrix, iteration_range=(0, model.best_iteration))` replaces `model.predict(X, num_iteration=model.best_iteration)`.

In [11]:
def wape(yt, yp):
    yt, yp = np.asarray(yt, float), np.asarray(yp, float)
    ok = ~(np.isnan(yt) | np.isnan(yp))
    yt, yp = yt[ok], yp[ok]
    d = np.abs(yt).sum()
    return float(np.abs(yp - yt).sum() / d) if d else 0.0

def wpe(yt, yp):
    yt, yp = np.asarray(yt, float), np.asarray(yp, float)
    ok = ~(np.isnan(yt) | np.isnan(yp))
    yt, yp = yt[ok], yp[ok]
    d = yt.sum()
    return float((yp - yt).sum() / d) if d else 0.0

# XGBoost custom_metric signature: (y_pred, dmatrix) -> (name, value)
# Note: REVERSED argument order vs LightGBM's feval(y_pred, dataset)
# XGBoost minimises the metric by default (maximize=False in xgb.train)
def xgb_wape_metric(y_pred, dmatrix):
    y_true = dmatrix.get_label()
    return "WAPE", wape(y_true, y_pred)

def plw(mu_vals):
    """Power-law weights (paper Section 3.2, alpha=2.83), capped at 10x."""
    mu  = np.asarray(mu_vals, float) + 1e-6
    raw = mu ** (1.0 / P.POWER_LAW_ALPHA)
    w   = raw / raw.mean()
    w   = np.clip(w, 0.1, 10.0)
    return w.astype(np.float32)

X_tr  = train_df[FEAT_COLS].fillna(0).values.astype(np.float32)
y_tr  = train_df["train_target"].values.astype(np.float32)
w_tr  = plw(train_df["feat_mu_i"].fillna(0).values)
X_val = val_df[FEAT_COLS].fillna(0).values.astype(np.float32)
y_val = val_df["train_target"].values.astype(np.float32)
w_val = plw(val_df["feat_mu_i"].fillna(0).values)

print(f"Train: {len(X_tr):,}  Val: {len(X_val):,}  Features: {len(FEAT_COLS)}")
print(f"y_tr  — mean:{y_tr.mean():.3f}  std:{y_tr.std():.3f}  min:{y_tr.min():.3f}")
print(f"y_val — mean:{y_val.mean():.3f}  std:{y_val.std():.3f}  min:{y_val.min():.3f}")

# Build DMatrix objects — XGBoost's equivalent of lgb.Dataset
# feature_names must be passed here so get_score() can align by name later
dtrain = xgb.DMatrix(X_tr,  label=y_tr,  weight=w_tr,  feature_names=FEAT_COLS)
dval   = xgb.DMatrix(X_val, label=y_val, weight=w_val, feature_names=FEAT_COLS)

# ── Why objective=reg:tweedie stays, but no eval_metric in params ────────
# objective=reg:tweedie: controls HOW trees are built — correct inductive
# bias for right-skewed zero-inflated perishable demand (alpha=2.83).
#
# We omit eval_metric from XGB_PARAMS so the built-in Tweedie log-likelihood
# is NOT used for early stopping (same reasoning as LGB v5's metric='None').
# The custom_metric=xgb_wape_metric drives both logging and early stopping.

t_train = time.time()
model = xgb.train(
    XGB_PARAMS,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=[(dval, "val"), (dtrain, "train")],
    custom_metric=xgb_wape_metric,     # drives early stopping; lower=better (minimize)
    maximize=False,                    # WAPE should be minimised
    early_stopping_rounds=EARLY_STOPPING,
    verbose_eval=VERBOSE_EVAL,
)
t_elapsed = time.time() - t_train

print(f"\nTraining done in {t_elapsed:.0f}s")
print(f"Best iteration : {model.best_iteration}")
print(f"Best val WAPE  : {model.best_score:.6f}")

Train: 330,406  Val: 79,449  Features: 62
y_tr  — mean:1.003  std:1.292  min:0.060
y_val — mean:1.187  std:1.872  min:0.060
[0]	val-tweedie-nloglik@1.5:5.06056	val-WAPE:0.72345	train-tweedie-nloglik@1.5:4.46583	train-WAPE:0.77129
[100]	val-tweedie-nloglik@1.5:4.44751	val-WAPE:0.29346	train-tweedie-nloglik@1.5:4.06846	train-WAPE:0.28112
[200]	val-tweedie-nloglik@1.5:4.44505	val-WAPE:0.28836	train-tweedie-nloglik@1.5:4.06338	train-WAPE:0.27048
[300]	val-tweedie-nloglik@1.5:4.44412	val-WAPE:0.28647	train-tweedie-nloglik@1.5:4.06048	train-WAPE:0.26444
[400]	val-tweedie-nloglik@1.5:4.44411	val-WAPE:0.28592	train-tweedie-nloglik@1.5:4.05850	train-WAPE:0.26020
[500]	val-tweedie-nloglik@1.5:4.44412	val-WAPE:0.28570	train-tweedie-nloglik@1.5:4.05689	train-WAPE:0.25656
[600]	val-tweedie-nloglik@1.5:4.44420	val-WAPE:0.28530	train-tweedie-nloglik@1.5:4.05548	train-WAPE:0.25338
[700]	val-tweedie-nloglik@1.5:4.44443	val-WAPE:0.28574	train-tweedie-nloglik@1.5:4.05423	train-WAPE:0.25048
[800]	val-twee

## 9 · Validation Metrics (Eq 4, 5)

In [12]:
# Predict using best_iteration — XGBoost uses iteration_range=(0, N)
val_preds = np.clip(
    model.predict(dval, iteration_range=(0, model.best_iteration)),
    0, None
)
w_val_score = wape(y_val, val_preds)
b_val_score = wpe(y_val, val_preds)

print(f"\n{'='*55}")
print(f"  Validation Metrics  (MNAR simulation set)")
print(f"{'='*55}")
print(f"  WAPE : {w_val_score*100:6.2f}%   (paper TimesNet best: 27.62%)")
print(f"  WPE  : {b_val_score*100:+6.2f}%   (target: |WPE| < 3%  |  raw: -7.37%)")
sym_wpe = "PASS" if abs(b_val_score) < P.TARGET_WPE else ("WARN-low" if b_val_score < 0 else "WARN-high")
print(f"  WPE status: {sym_wpe}")


  Validation Metrics  (MNAR simulation set)
  WAPE :  29.24%   (paper TimesNet best: 27.62%)
  WPE  : -11.78%   (target: |WPE| < 3%  |  raw: -7.37%)
  WPE status: WARN-low


In [13]:
# Bias calibration on in-stock val rows
# Identical logic to LightGBM v5 — scalar correction factor on in-stock rows.

in_stock_val_mask = val_df[C.IN_STOCK].values == 1
y_is   = y_val[in_stock_val_mask]
p_is   = val_preds[in_stock_val_mask]

calib_factor = float(y_is.sum() / (p_is.sum() + 1e-9))
calib_factor = np.clip(calib_factor, 0.80, 1.25)

val_preds_cal = val_preds * calib_factor

w_val_score_cal = wape(y_val, val_preds_cal)
b_val_score_cal = wpe(y_val,  val_preds_cal)

print(f"Calibration factor      : {calib_factor:.4f}")
print(f"Pre-calib  WAPE: {w_val_score*100:.2f}%   WPE: {b_val_score*100:+.2f}%")
print(f"Post-calib WAPE: {w_val_score_cal*100:.2f}%   WPE: {b_val_score_cal*100:+.2f}%")

# Use calibrated predictions for all downstream work
val_preds    = val_preds_cal
w_val_score  = w_val_score_cal
b_val_score  = b_val_score_cal

Calibration factor      : 1.1335
Pre-calib  WAPE: 29.24%   WPE: -11.78%
Post-calib WAPE: 28.52%   WPE: +0.00%


In [14]:
# Reconstruct stockout indicator for evaluation
df["feat_so_frac"] = (df[C.IN_STOCK] == 0).astype(np.float32)

## 10 · Demand Reconstruction (Eq 2) + ρ_DS (Eq 6)

In [15]:
# Predict on full dataset
# XGBoost requires a DMatrix; build one from full feature matrix
X_full  = df[FEAT_COLS].fillna(0).values.astype(np.float32)
dfull   = xgb.DMatrix(X_full, feature_names=FEAT_COLS)
D_hat   = np.clip(
    model.predict(dfull, iteration_range=(0, model.best_iteration)),
    0, None
)
D_hat   = D_hat * calib_factor

# Eq 2 (daily): d_t = y_t*s_t + d_hat_t*(1-s_t)
s   = df[C.IN_STOCK].values.astype(np.float32)
y   = df[C.SALE_AMOUNT].values.astype(np.float32)
D_t = y * s + D_hat * (1.0 - s)

df[C.D_HAT] = D_hat.astype(np.float32)
df[C.D_T]   = D_t.astype(np.float32)

obs_vol = float((y*s).sum())
imp_vol = float((D_hat*(1-s)).sum())
print(f"=== Eq 2 Reconstruction ===")
print(f"  In-stock days   : {int(s.sum()):>10,} -> keep observed y_t")
print(f"  Stockout days   : {int((1-s).sum()):>10,} -> use model D_hat_t")
print(f"  Observed volume : {obs_vol:>14,.1f}")
print(f"  Imputed volume  : {imp_vol:>14,.1f}  (+{100*imp_vol/max(obs_vol,1):.1f}%)")
print(f"  Total recovered : {obs_vol+imp_vol:>14,.1f}")

df["recovery_uplift"] = np.where(y > 0, D_t / y, 1.0)
df["recovery_uplift"] = df["recovery_uplift"].clip(0, 10)

print(f"\nRecovery uplift (stockout days only):")
so_up = df.loc[df[C.IN_STOCK]==0, "recovery_uplift"]
print(f"  mean: {so_up.mean():.3f}x  median: {so_up.median():.3f}x  (expected 1.1-1.5x)")

=== Eq 2 Reconstruction ===
  In-stock days   :  2,780,661 -> keep observed y_t
  Stockout days   :  1,719,339 -> use model D_hat_t
  Observed volume :    2,828,200.8
  Imputed volume  :    2,218,439.1  (+78.4%)
  Total recovered :    5,046,639.9

Recovery uplift (stockout days only):
  mean: 1.452x  median: 1.042x  (expected 1.1-1.5x)


In [16]:
df["feat_so_frac"] = (df[C.IN_STOCK] == 0).astype(np.float32)

In [17]:
# ρ_DS — pair-level Pearson (Eq 6)
pair_stats = df.groupby([C.STORE_ID, C.PRODUCT_ID]).agg(
    SR_i  = ("feat_so_frac", "mean"),
    d_i   = (C.D_T,  "mean"),
    mu_i  = (C.MU_I, "first"),
).reset_index()
pair_stats = pair_stats[pair_stats["mu_i"] > 0].copy()
pair_stats["w_i"] = pair_stats["mu_i"] / pair_stats["mu_i"].sum()

mu_sr = (pair_stats["w_i"] * pair_stats["SR_i"]).sum()
mu_di = (pair_stats["w_i"] * pair_stats["d_i"]).sum()
num   = (pair_stats["w_i"]*(pair_stats["SR_i"]-mu_sr)*(pair_stats["d_i"]-mu_di)).sum()
s_sr  = np.sqrt((pair_stats["w_i"]*(pair_stats["SR_i"]-mu_sr)**2).sum())
s_di  = np.sqrt((pair_stats["w_i"]*(pair_stats["d_i"] -mu_di)**2).sum())
rho_ds = float(num / (s_sr * s_di + 1e-9))

sym = "PASS" if abs(rho_ds) < P.TARGET_RHO_DS else "FAIL"
print(f"rho_DS (pair-level, Eq 6) = {rho_ds:+.4f}  [{sym}]")
print(f"  Pairs: {len(pair_stats):,}  |  target |rho_DS|<{P.TARGET_RHO_DS}")
print(f"  Paper raw=-0.57  TimesNet best=+0.07")

rho_DS (pair-level, Eq 6) = +0.4630  [FAIL]
  Pairs: 50,000  |  target |rho_DS|<0.1
  Paper raw=-0.57  TimesNet best=+0.07


## 11 · Save Outputs

In [18]:
save_cols = [c for c in [
    C.CITY_ID, C.STORE_ID, C.MGMT_GRP,
    C.CAT1, C.CAT2, C.CAT3, C.PRODUCT_ID, C.DT,
    C.SALE_AMOUNT, C.IN_STOCK, C.D_HAT, C.D_T, "recovery_uplift",
    C.MU_I, C.SR_I,
    C.DISCOUNT, C.ACTIVITY, C.HOLIDAY,
    C.PRECPT, C.TEMP, C.HUMID, C.WIND,
    "first_stockout_hour", "stockout_in_peak",
    "feat_dow", "feat_is_wkend", "feat_month",
] if c in df.columns]

out_path = OUTPUT_DIR / "stage1_daily_D_t_train.parquet"
df[save_cols].to_parquet(out_path, index=False)
print(f"Saved: {out_path}  ({out_path.stat().st_size/1e6:.1f} MB)")

# Save model
model_path = OUTPUT_DIR / "stage1_xgb_model.json"
model.save_model(str(model_path))
print(f"Model saved: {model_path}")

# Save calibration factor for Stage 2
calib_path = OUTPUT_DIR / "calib_factor.pkl"
with open(calib_path, "wb") as f:
    pickle.dump({"calib_factor": calib_factor, "feat_cols": FEAT_COLS}, f)
print(f"Calib factor saved: {calib_path}")

Saved: /kaggle/working/stage1_xgb_v1/stage1_daily_D_t_train.parquet  (69.6 MB)
Model saved: /kaggle/working/stage1_xgb_v1/stage1_xgb_model.json
Calib factor saved: /kaggle/working/stage1_xgb_v1/calib_factor.pkl


## 12 · Eval on eval.parquet

In [19]:
print("Loading eval ...")
df_eval = load_daily(EVAL_PATH, "eval")
df_eval = build_features(df_eval, is_train=False)

for c in FEAT_COLS:
    if c not in df_eval.columns: df_eval[c] = 0.0

X_e     = df_eval[FEAT_COLS].fillna(0).values.astype(np.float32)
de_mat  = xgb.DMatrix(X_e, feature_names=FEAT_COLS)
D_hat_e = np.clip(
    model.predict(de_mat, iteration_range=(0, model.best_iteration)),
    0, None
) * calib_factor

s_e   = df_eval[C.IN_STOCK].values.astype(np.float32)
y_e   = df_eval[C.SALE_AMOUNT].values.astype(np.float32)
D_t_e = y_e * s_e + D_hat_e * (1.0 - s_e)
df_eval[C.D_T] = D_t_e.astype(np.float32)
df_eval["recovery_uplift"] = np.where(y_e > 0, D_t_e/y_e, 1.0).clip(0, 10)

eval_save = [c for c in save_cols if c in df_eval.columns]
out_eval  = OUTPUT_DIR / "stage1_daily_D_t_eval.parquet"
df_eval[eval_save].to_parquet(out_eval, index=False)
print(f"Eval D_t -> {out_eval}  ({out_eval.stat().st_size/1e6:.1f} MB)")
print(f"Uplift mean: {df_eval['recovery_uplift'].mean():.3f}x")

Loading eval ...
  Vectorised array parsing on 350,000 rows ...


10:25:09  INFO      [eval] 350,000 rows in 4.4s  RAM:7987MB
10:25:09  INFO        Date: 2024-06-26 to 2024-07-02
10:25:09  INFO        Stores: 898  Products: 865
10:25:09  INFO        In-stock days (stockout_hrs <= 2): 65.0%
10:29:12  INFO      Features built: 62 in 243.4s  RAM:8009MB


Eval D_t -> /kaggle/working/stage1_xgb_v1/stage1_daily_D_t_eval.parquet  (4.8 MB)
Uplift mean: 1.110x


## 13 · Diagnostic Plots (9-panel)

Identical to LightGBM v5, with one change: feature importance is extracted via
`model.get_score(importance_type='gain')` which returns a **dict** keyed by feature
name. We align it to `FEAT_COLS` order manually (missing features get 0 gain).

In [20]:
import matplotlib
matplotlib.use("Agg")

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle("Stage 1 — XGBoost Demand Recovery (v1): All Diagnostics",
             fontsize=14, fontweight="bold", y=0.98)

# A: Predicted vs actual (val set)
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y_val, val_preds, alpha=0.15, s=4, color="#185FA5")
lim = max(float(y_val.max()), float(val_preds.max())) * 1.05
ax1.plot([0, lim], [0, lim], "r--", lw=1.2, label="Perfect recovery")
ax1.set_xlabel("Actual demand (ground truth)")
ax1.set_ylabel("XGB predicted demand")
ax1.set_title(f"A. Predicted vs actual\nWAPE={w_val_score*100:.2f}%  WPE={b_val_score*100:+.2f}%")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# B: Residual distribution
ax2 = fig.add_subplot(gs[0, 1])
res = val_preds - y_val
ax2.hist(res, bins=60, color="#185FA5", edgecolor="white", alpha=0.8)
ax2.axvline(0,         color="red",    lw=1.5, ls="--", label="Zero bias")
ax2.axvline(res.mean(),color="orange", lw=1.5, ls="--", label=f"Mean={res.mean():.3f}")
ax2.set_xlabel("Residual (predicted - actual)")
ax2.set_title("B. Residual distribution\n(symmetric around 0 = unbiased)")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# C: Feature importance
# XGBoost get_score() returns dict; align to FEAT_COLS order (0 for unsplit features)
ax3 = fig.add_subplot(gs[0, 2])
gain_dict = model.get_score(importance_type="gain")
gain_vals = np.array([gain_dict.get(f, 0.0) for f in FEAT_COLS], dtype=np.float32)
fi = pd.DataFrame({"feature": FEAT_COLS, "gain": gain_vals}
                  ).sort_values("gain", ascending=True).tail(20)
supply_kw = {"so_", "stockout", "in_stock", "stock_frac", "hours_in"}
def is_supply(f): return any(k in f for k in supply_kw)
colors3 = ["#E24B4A" if is_supply(f) else "#185FA5" for f in fi["feature"]]
ax3.barh(fi["feature"], fi["gain"], color=colors3, alpha=0.85)
ax3.set_xlabel("Split Gain")
ax3.set_title("C. Feature importance\n(red=supply, blue=demand)")
ax3.legend(handles=[Patch(facecolor="#E24B4A", label="Supply"),
                    Patch(facecolor="#185FA5", label="Demand/other")],
           fontsize=7, loc="lower right")
ax3.tick_params(axis="y", labelsize=7); ax3.grid(axis="x", alpha=0.3)

# D: Weekly pattern raw vs recovered
ax4 = fig.add_subplot(gs[1, 0])
raw_wd = df.groupby("feat_dow")[C.SALE_AMOUNT].mean()
rec_wd = df.groupby("feat_dow")[C.D_T].mean()
days = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
ax4.plot(days, raw_wd.values[:7], color="#E24B4A", lw=2, marker="o", ms=5, label="Raw (censored)")
ax4.plot(days, rec_wd.values[:7], color="#0F6E56", lw=2, marker="s", ms=5, label="Recovered D_t")
ax4.set_xlabel("Day of week"); ax4.set_ylabel("Mean demand")
ax4.set_title("D. Weekly pattern: raw vs recovered\n(recovered must be >= raw on average)")
ax4.legend(fontsize=8); ax4.grid(alpha=0.3)

# E: Recovery uplift — stockout vs in-stock days
ax5 = fig.add_subplot(gs[1, 1])
up_so = df.loc[df[C.IN_STOCK]==0, "recovery_uplift"].clip(0, 5)
up_is = df.loc[df[C.IN_STOCK]==1, "recovery_uplift"].clip(0, 3)
ax5.hist(up_so, bins=40, color="#E24B4A", alpha=0.7, label=f"Stockout (mean={up_so.mean():.2f}x)", density=True)
ax5.hist(up_is, bins=40, color="#1D9E75", alpha=0.5, label=f"In-stock (mean={up_is.mean():.2f}x)", density=True)
ax5.axvline(1.0, color="gray", lw=1.5, ls="--", label="No correction")
ax5.set_xlabel("D_t / raw_sales")
ax5.set_title("E. Recovery uplift by day type\n(stockout days should be above 1.0x)")
ax5.legend(fontsize=8); ax5.grid(alpha=0.3)

# F: Pair-level rho_DS scatter (Eq 6)
ax6 = fig.add_subplot(gs[1, 2])
smp = pair_stats.sample(min(3000, len(pair_stats)), random_state=42)
sc6 = ax6.scatter(smp["SR_i"], smp["d_i"],
                  c=np.log1p(smp["mu_i"]), cmap="viridis", alpha=0.35, s=6)
plt.colorbar(sc6, ax=ax6, label="log(mu_i)")
z6 = np.polyfit(smp["SR_i"], smp["d_i"], 1)
x6 = np.linspace(smp["SR_i"].min(), smp["SR_i"].max(), 100)
ax6.plot(x6, np.polyval(z6,x6), "r--", lw=1.5)
ax6.set_xlabel("SR_i (stockout fraction per pair)")
ax6.set_ylabel("d_i (mean recovered demand)")
ax6.set_title(f"F. Pair-level rho_DS (Eq 6)\nrho={rho_ds:+.4f}  target|rho|<{P.TARGET_RHO_DS}")
ax6.grid(alpha=0.3)

# G: Single series recovery timeline
ax7 = fig.add_subplot(gs[2, 0:2])
series_stats = df.groupby("feat_store_id").agg(
    n_so=(C.IN_STOCK, lambda x: (x==0).sum())
).query("n_so >= 5")
if len(series_stats) > 0:
    sid = series_stats.index[0]
    store_df = df[df["feat_store_id"] == sid].groupby("feat_product_id").agg(
        n_so=(C.IN_STOCK, lambda x: (x==0).sum())
    ).query("n_so >= 5")
    if len(store_df) > 0:
        pid    = store_df.index[0]
        ser_df = df[(df["feat_store_id"]==sid) & (df["feat_product_id"]==pid)].sort_values(C.DT)
        ax7.bar(ser_df[C.DT], ser_df[C.SALE_AMOUNT], width=1, color="#185FA5", alpha=0.6, label="Observed (y_t)")
        ax7.bar(ser_df.loc[ser_df[C.IN_STOCK]==0, C.DT],
                ser_df.loc[ser_df[C.IN_STOCK]==0, C.D_T],
                width=1, color="#E24B4A", alpha=0.8, label="Recovered D_t (stockout days)")
        ax7.set_xlabel("Date"); ax7.set_ylabel("Demand")
        ax7.set_title("G. Demand recovery for one series\n(red bars = imputed stockout demand)")
        ax7.legend(fontsize=8); ax7.grid(alpha=0.3)

# H: Stockout impact diagnostics (multi-view)
inner_gs = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[2, 2], wspace=0.4)
h_df = df.loc[val_df.index].copy()
h_df["pred"] = val_preds
h_df["err"]  = (val_preds - y_val) / (y_val + 1e-8)

ax_h1 = fig.add_subplot(inner_gs[0, 0])
so_hours = h_df["_tmp_so_hours"]
bins_h1  = pd.cut(so_hours, bins=5)
err_by_so = h_df.groupby(bins_h1)["err"].mean()
ax_h1.bar(range(len(err_by_so)), err_by_so.values, color="#EF9F27", alpha=0.8)
ax_h1.axhline(0, color="gray", ls="--", lw=1)
ax_h1.set_title("H1. Error vs stockout hours", fontsize=9)
ax_h1.set_xticks(range(len(err_by_so)))
ax_h1.set_xticklabels([str(b) for b in err_by_so.index], rotation=30, fontsize=6)
ax_h1.grid(axis="y", alpha=0.3)

ax_h2 = fig.add_subplot(inner_gs[0, 1])
first_so  = h_df["_tmp_first_so_hour"]
bins_h2   = pd.cut(first_so, bins=5)
err_by_first = h_df.groupby(bins_h2)["err"].mean()
ax_h2.bar(range(len(err_by_first)), err_by_first.values, color="#2A9D8F", alpha=0.8)
ax_h2.axhline(0, color="gray", ls="--", lw=1)
ax_h2.set_title("H2. Error vs first SO hour", fontsize=9)
ax_h2.set_xticks(range(len(err_by_first)))
ax_h2.set_xticklabels([str(b) for b in err_by_first.index], rotation=30, fontsize=6)
ax_h2.grid(axis="y", alpha=0.3)

ax_h3 = fig.add_subplot(inner_gs[0, 2])
peak     = h_df["_tmp_so_in_peak"]
err_peak = h_df.loc[peak == 1, "err"]
err_non  = h_df.loc[peak == 0, "err"]
ax_h3.boxplot([err_non, err_peak], labels=["Non-peak", "Peak"])
ax_h3.axhline(0, color="gray", ls="--", lw=1)
ax_h3.set_title("H3. Peak vs non-peak SO", fontsize=9)
ax_h3.grid(axis="y", alpha=0.3)

plt.savefig(OUTPUT_DIR/"stage1_diagnostics_xgb_v1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Diagnostic plot saved.")

10:29:32  INFO      Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
10:29:32  INFO      Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
10:29:32  INFO      Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
10:29:32  INFO      Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Diagnostic plot saved.


In [21]:
import matplotlib
matplotlib.use("Agg")

fig2 = plt.figure(figsize=(16, 8))
gs2  = gridspec.GridSpec(2, 1, figure=fig2, hspace=0.35)

fig2.suptitle("Demand Recovery Visualization (XGBoost)",
              fontsize=14, fontweight="bold", y=0.98)
ax1 = fig2.add_subplot(gs2[0, 0])

# Pick a store-product pair with at least 5 stockout days for visualisation
series_stats2 = df.groupby("feat_store_id").agg(
    n_so=(C.IN_STOCK, lambda x: (x==0).sum())
).query("n_so >= 5")
if len(series_stats2) > 0:
    sid2 = series_stats2.index[0]
    store_df2 = df[df["feat_store_id"]==sid2].groupby("feat_product_id").agg(
        n_so=(C.IN_STOCK, lambda x: (x==0).sum())
    ).query("n_so >= 5")
    if len(store_df2) > 0:
        pid2   = store_df2.index[0]
        ser_df = df[(df["feat_store_id"]==sid2) & (df["feat_product_id"]==pid2)].sort_values(C.DT)

        ax1.plot(ser_df[C.DT], ser_df[C.SALE_AMOUNT], color="#E24B4A", lw=2, label="Observed (censored)")
        ax1.plot(ser_df[C.DT], ser_df[C.D_T],         color="#0F6E56", lw=2, label="Recovered demand")

        so_mask = ser_df[C.IN_STOCK] == 0
        for dt in ser_df.loc[so_mask, C.DT]:
            ax1.axvspan(dt, dt, color="#E24B4A", alpha=0.15)
        ax1.scatter(ser_df.loc[so_mask, C.DT], ser_df.loc[so_mask, C.D_T],
                    color="#0F6E56", marker="^", s=40)

        ax1.set_title("Observed vs recovered demand\n(red shading = stockouts)")
        ax1.set_ylabel("Demand"); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

        ax2v = fig2.add_subplot(gs2[1, 0])
        ser_df2 = ser_df.copy()
        ser_df2["gap"] = ser_df2[C.D_T] - ser_df2[C.SALE_AMOUNT]
        gap_df = ser_df2[ser_df2[C.IN_STOCK] == 0]
        ax2v.bar(gap_df[C.DT], gap_df["gap"], color="#0F6E56", alpha=0.8)
        ax2v.axhline(0, color="gray", lw=1.2, ls="--")
        ax2v.set_title("Recovered demand gap on stockout days")
        ax2v.set_xlabel("Date"); ax2v.set_ylabel("Recovered - observed")
        ax2v.grid(axis="y", alpha=0.3)

plt.savefig(OUTPUT_DIR/"stage1_recovery_visualization_xgb.png", dpi=150, bbox_inches="tight")
plt.show()
print("Recovery visualization saved.")

Recovery visualization saved.


## 14 · Final Summary

In [22]:
print("\n" + "="*60)
print("  STAGE 1 XGBoost v1 COMPLETE")
print("="*60)
print(f"  Val WAPE : {w_val_score*100:.2f}%   (paper TimesNet: 27.62%)")
print(f"  Val WPE  : {b_val_score*100:+.2f}%   (target |WPE|<3%  raw=-7.37%)")
print(f"  rho_DS   : {rho_ds:+.4f}   (pair-level Eq 6)")
print(f"  Uplift   : {df.loc[df[C.IN_STOCK]==0,'recovery_uplift'].mean():.3f}x (stockout days)")
print(f"  Trees    : {model.num_boosted_rounds()}")
print(f"  Best itr : {model.best_iteration}")
print(f"  RAM      : {mem_mb():.0f} MB")
print()
print("  XGBoost vs LightGBM translation summary:")
print("    objective        : reg:tweedie (power=1.5)  ← lgb tweedie (power=1.5)")
print("    metric           : custom WAPE via custom_metric= ← lgb feval=")
print("    early_stopping   : early_stopping_rounds arg ← lgb callbacks")
print("    feature_names    : passed to DMatrix constructor")
print("    feature imp      : get_score(type='gain') dict ← feature_importance(type='gain') array")
print("    predict          : predict(dmat, iteration_range=(0,N)) ← predict(X, num_iteration=N)")
print("    model save       : model.save_model('.json') ← pickle")
print()
print("  Outputs:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"    {f.name:<55s}  {f.stat().st_size/1e6:.1f} MB")
print()
print("  stage1_daily_D_t_train.parquet -> Stage 2 forecasting input")


  STAGE 1 XGBoost v1 COMPLETE
  Val WAPE : 28.52%   (paper TimesNet: 27.62%)
  Val WPE  : +0.00%   (target |WPE|<3%  raw=-7.37%)
  rho_DS   : +0.4630   (pair-level Eq 6)
  Uplift   : 1.452x (stockout days)
  Trees    : 4000
  Best itr : 3999
  RAM      : 7784 MB

  XGBoost vs LightGBM translation summary:
    objective        : reg:tweedie (power=1.5)  ← lgb tweedie (power=1.5)
    metric           : custom WAPE via custom_metric= ← lgb feval=
    early_stopping   : early_stopping_rounds arg ← lgb callbacks
    feature_names    : passed to DMatrix constructor
    feature imp      : get_score(type='gain') dict ← feature_importance(type='gain') array
    predict          : predict(dmat, iteration_range=(0,N)) ← predict(X, num_iteration=N)
    model save       : model.save_model('.json') ← pickle

  Outputs:
    calib_factor.pkl                                         0.0 MB
    stage1_daily_D_t_eval.parquet                            4.8 MB
    stage1_daily_D_t_train.parquet            